# Greyscale ablation — Variant C / Design C1 (orchestrator)

**Isolated experiment on branch `greyscale-experiment`.** Adds NEW files only; touches
no completed Phase 1–4 config, module, checkpoint, report, or figure. All artefacts land
under `greyscale_experiment/` on Drive.

## What Design C1 tests
The completed study found zero-shot cross-modality DR grading **collapses** (96% grade 0 on
OLIVES), and the confident "severe" tail is an **image-brightness artifact**. Design C1 asks:
*if we harmonise the source toward the target intensity distribution, does the collapse ease?*

**Variant C transform (EyePACS only):** green channel → histogram-match to the OLIVES near-IR
intensity distribution → replicate to 3 identical channels. **Design C1:** transform EyePACS
only; OLIVES left **raw** at train and eval. Labels unchanged (image-only transform).

Flow: setup → (1) reference histogram → (2) pre-flight + (3) label-preservation → (4) train →
(5) evaluate (unmodified Phase 4a module) → (6) compare vs the original collapse.

## STEP 1 — how Phase 2 training runs, and where the transform hooks in

**Training entry (Phase 2):** `src/training/train.py` (`python -m src.training.train --config
configs/train.yaml`). Merges `model.yaml`+`data.yaml`+`train.yaml`, seeds 42, builds loaders via
`build_dataloaders`, runs `Trainer.fit()` (AdamW, warmup→cosine, AMP, `combined_loss`; model
selection on validation DR QWK). `Trainer._ckpt_path` writes `phase2_{run_name}_{best,last}.pt`.

**Harmonised entry (this experiment):** `scripts/train_greyscale.py` reuses that exact logic
and inserts ONLY the EyePACS-only harmonisation after `build_dataloaders` via
`harmonise_eyepacs_in_bundle` (`run_name: greenhistmatch`,
`checkpoint.dir=greyscale_experiment` → `phase2_greenhistmatch_best.pt`; never the original).

**Transform insertion point:** `EyePACSDataset.__getitem__` applies `self.transform(image)` then
returns `dr_labels` from the immutable `self._labels` vector — labels are independent of pixels,
so the transform cannot alter them. Stored tensors are ImageNet-normalised, so `GreenHistMatch`
de-normalises → green + hist-match + 3ch replicate → re-normalises, before the existing aug.

**Eval (unchanged Phase 4a):** `src/analysis/zero_shot_grading.py` reads the checkpoint's own
config and feeds **raw OLIVES**; pointing `configs/greyscale_eval.yaml` at the new checkpoint +
`greyscale_experiment/` output dirs is sufficient — no code change.

In [ ]:
# Setup: mount Drive, restore the repo on the greyscale-experiment branch, cd in.
from google.colab import drive
import os

drive.mount('/content/drive')

REPO_DIR = '/content/dr-dissertation'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/savita10/dr-dissertation.git {REPO_DIR}
%cd {REPO_DIR}
!git fetch origin && git checkout greyscale-experiment && git pull

import torch
print('CUDA available:', torch.cuda.is_available())

## 1. OLIVES near-IR reference histogram
One-off, CPU-fine, idempotent. Writes `greyscale_experiment/olives_reference_hist.npy`
(green-channel intensity CDF in [0,1]). Re-running just summarises the existing file.

In [ ]:
!python -m scripts.compute_olives_histogram --config configs/greyscale_train.yaml

## 2. Pre-flight checks (STEP 1)
Confirm branch, that the reference histogram exists, and that a GPU is available.
Do not proceed to training if any fails.

In [ ]:
import subprocess
from pathlib import Path
import torch
from src.utils.config import load_config

branch = subprocess.run(
    ['git', 'rev-parse', '--abbrev-ref', 'HEAD'], capture_output=True, text=True
).stdout.strip()
print('branch:', branch)
assert branch == 'greyscale-experiment', f'expected greyscale-experiment, on {branch}'

ref = Path(str(load_config('configs/greyscale_train.yaml').reference_hist))
print('reference histogram:', ref, '| exists =', ref.exists())
assert ref.exists(), 'run the histogram cell (section 1) first'

print('CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'GPU required for training + eval — switch to a GPU runtime'
print('GPU:', torch.cuda.get_device_name(0))

## 3. Label-preservation verification (MANDATORY — re-assert before training)
1. sample batch **with vs without** harmonise → labels identical element-wise;
2. per-grade counts (0–4) over the **full** harmonised train split == original counts;
3. before/after thumbnails. If (1) or (2) fail, **STOP** — labels not preserved.

In [ ]:
import numpy as np
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from src.training.train import build_merged_config
from src.data.dataloaders import build_dataloaders, collate_fn
from src.preprocessing.harmonise import GreenHistMatch, harmonise_eyepacs_in_bundle

cfg = build_merged_config('configs/greyscale_train.yaml')
bundle = build_dataloaders(cfg)

ep_ds = bundle['train_loader'].dataset.datasets[0]  # EyePACS train sub-dataset
train_idx = bundle['splits']['eyepacs']['train']
orig_labels = ep_ds.labels[torch.as_tensor(list(train_idx))].long()
orig_counts = torch.bincount(orig_labels, minlength=5)
print('EyePACS train split size:', len(ep_ds))
print('original per-grade counts (0-4):', orig_counts.tolist())

In [ ]:
# Capture RAW normalised tensors (transform off) for clean thumbnails + a
# confound-free pixel-change check, then restore the original transform.
sample_ids = list(range(6))
orig_tf = ep_ds.transform
ep_ds.transform = None
raw_norm = [ep_ds[i]['images'].clone() for i in sample_ids]
raw_labels = [ep_ds[i]['dr_labels'] for i in sample_ids]
ep_ds.transform = orig_tf

ghm = GreenHistMatch(
    cfg.reference_hist, cfg.imagenet_mean, cfg.imagenet_std, int(cfg.reference_channel)
)
harm_norm = [ghm(t) for t in raw_norm]
pixel_delta = float(torch.stack([(h - r).abs().mean() for h, r in zip(harm_norm, raw_norm)]).mean())
print('mean abs pixel change (harmonised vs colour):', pixel_delta)
assert pixel_delta > 0.0, 'transform did not change pixels — unexpected'

In [ ]:
# (1) Sample batch WITH vs WITHOUT harmonise -> labels identical element-wise.
sample_n = 128
labels_before = torch.tensor([ep_ds[i]['dr_labels'] for i in range(sample_n)])

n_wrapped = harmonise_eyepacs_in_bundle(
    bundle, cfg.reference_hist, cfg.imagenet_mean, cfg.imagenet_std, int(cfg.reference_channel)
)
print('EyePACS datasets harmonised (train/val/test):', n_wrapped)

labels_after = torch.tensor([ep_ds[i]['dr_labels'] for i in range(sample_n)])
assert torch.equal(labels_before, labels_after), 'LABELS CHANGED — STOP (labels not preserved)'
print('sample-batch labels identical with vs without harmonise:', True)

In [ ]:
# (2) Full harmonised train split -> per-grade counts must equal the original.
ep_loader = DataLoader(
    ep_ds, batch_size=256, shuffle=False, collate_fn=collate_fn, num_workers=2
)
harm_counts = torch.zeros(5, dtype=torch.long)
for b in tqdm(ep_loader, desc='harmonised EyePACS train'):
    harm_counts += torch.bincount(b['dr_labels'], minlength=5)

print('original    per-grade counts (0-4):', orig_counts.tolist())
print('harmonised  per-grade counts (0-4):', harm_counts.tolist())
assert torch.equal(harm_counts, orig_counts), (
    'PER-GRADE COUNTS DIFFER — STOP (labels not preserved over the full split)'
)
print('\nLABEL PRESERVATION VERIFIED: image-only transform, labels + split intact.')

In [ ]:
# (3) Before/after thumbnails: colour EyePACS vs green-hist-matched.
import matplotlib.pyplot as plt

mean = torch.tensor(list(cfg.imagenet_mean)).view(3, 1, 1)
std = torch.tensor(list(cfg.imagenet_std)).view(3, 1, 1)

def denorm(t):
    return (t * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()

n = len(sample_ids)
fig, axes = plt.subplots(2, n, figsize=(2.4 * n, 5))
for j, (r, h, lab) in enumerate(zip(raw_norm, harm_norm, raw_labels)):
    axes[0, j].imshow(denorm(r)); axes[0, j].axis('off')
    axes[0, j].set_title(f'colour (grade {lab})', fontsize=9)
    axes[1, j].imshow(denorm(h)); axes[1, j].axis('off')
    axes[1, j].set_title('green + hist-match', fontsize=9)
fig.suptitle('EyePACS source: colour (top) vs Design C1 harmonised (bottom)', y=1.02)
plt.tight_layout(); plt.show()

## 4. Train — Design C1 harmonised model
Same Phase 2 logic (seed 42, same architecture/heads/losses/optimizer/schedule/splits) via
`scripts/train_greyscale.py`, differing ONLY in EyePACS harmonisation and the output path
`greyscale_experiment/phase2_greenhistmatch_best.pt`. Smoke gate first (`[1/6]…[6/6]`), then the
full run (resumable — re-run the full cell after a disconnect; it continues from `_last.pt`).

In [ ]:
# Smoke gate — expect [1/6]…[6/6] pass and exit code 0.
!python -m scripts.train_greyscale --config configs/greyscale_train.yaml --smoke

In [ ]:
# Full harmonised run. Prints best validation DR QWK vs Phase 2 (~0.615) at the end.
!python -m scripts.train_greyscale --config configs/greyscale_train.yaml

## 5. Evaluate — zero-shot on OLIVES (UNMODIFIED Phase 4a module)
Runs the existing `src.analysis.zero_shot_grading` via `configs/greyscale_eval.yaml`: the new
checkpoint, raw OLIVES, outputs written under `greyscale_experiment/` (predictions CSV, grade
distribution, confidence/entropy figures). The original OLIVES predictions/report are untouched.

In [ ]:
# Save-safety (pre-flight): Drive does not auto-create parents. Ensure every eval
# output dir exists + the harmonised checkpoint is present before running eval.
from pathlib import Path
from src.utils.config import load_config

ev = load_config('configs/greyscale_eval.yaml')
for key in ('features_dir', 'report_dir', 'figures_dir'):
    d = Path(str(ev[key]))
    d.mkdir(parents=True, exist_ok=True)
    print(f'{key}: {d} | exists = {d.exists()}')
ckpt = Path(str(ev.checkpoint))
print(f'checkpoint: {ckpt} | exists = {ckpt.exists()}')
assert ckpt.exists(), 'train the harmonised model (section 4) first'

In [ ]:
!python -m src.analysis.zero_shot_grading --config configs/greyscale_eval.yaml

## 6. Compare vs the original 96% collapse
Writes `greyscale_experiment/greyscale_comparison.json`, `greyscale_report.md`, and a
side-by-side grade-distribution figure, with the honest pre-agreed interpretation frame.

In [ ]:
!python -m scripts.greyscale_compare --eval-config configs/greyscale_eval.yaml

In [ ]:
from IPython.display import Markdown, Image, display
from pathlib import Path

root = Path('/content/drive/MyDrive/dissertation/greyscale_experiment')
display(Markdown((root / 'greyscale_report.md').read_text()))
display(Image(str(root / 'figures' / 'greyscale_grade_distribution_compare.png')))

## Done
Artefacts under `greyscale_experiment/`: `olives_reference_hist.npy`,
`phase2_greenhistmatch_best.pt` (+ `_last.pt`, `_history.json`), `features/tta_olives.pt`,
`reports/` (phase4a summary + predictions CSV), `figures/`, `greyscale_comparison.json`,
`greyscale_report.md`. Nothing overwrites `phase2_coral_on_best.pt` or any Phase 1–4 output.

**Interpretation is honest and staged:** a spread distribution is *not* validated grading —
concordance with OLIVES clinical labels (CST / biomarker burden, as in Phase 4b) is the next
check. Whether this is merged or written up as future work depends on the result and supervisor
input. Commit/push happens from PowerShell on the `greyscale-experiment` branch.